# 08 — Predictive Customer Value

The available Olist history supports shorter future paid-value targets. This notebook estimates future customer-paid order value over 90, 180 and 365 days. It does not estimate profit or contribution margin.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.data_loader import load_olist_data
from src.analytics import prepare_order_level
from src.clv import run_clv_models

data = load_olist_data(ROOT / "data_raw")
order_level = prepare_order_level(data)
analysis_end = pd.Timestamp("2018-08-31 23:59:59")



## Train separate horizon models

Each horizon uses historical snapshots with complete future follow-up. The evaluation uses the latest snapshot dates as a chronological holdout. Compare each model with the median baseline; do not interpret a model as useful merely because it runs.


In [ ]:
value_predictions, value_metrics, value_importance = run_clv_models(
    order_level,
    analysis_end=analysis_end,
    model_dir=str(ROOT / "models")
)

value_metrics



In [ ]:
value_predictions.head()



In [ ]:
value_importance.groupby("horizon_days").head(10)



## Horizon limitation

- 90 days: short-term predicted paid value.
- 180 days: medium-term predicted paid value.
- 365 days: one-year predicted paid value, if sufficient training snapshots exist.
- 730 days: not estimated. The dataset does not contain sufficient reliable follow-up to train and validate a 24-month outcome.

The output is not contribution-margin CLV. To estimate economic CLV, the project would need product cost, returns/refunds, fulfilment, payment, service and acquisition costs.
